In [1]:
import sys
import torch
import os
from transformers import AutoTokenizer, GPT2Tokenizer
import pickle
import pandas as pd
import platform
from torch.nn import functional as F
from tqdm.auto import tqdm
from huggingface_hub import HfApi, create_repo

import sqlite3

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from huggingface_hub import login as hf_login
hf_login(token=os.getenv("HF_ACCESS_TOKEN"))

Token will not been saved to git credential helper. Pass `add_to_git_credential=True` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /home/abishekthamma/.cache/huggingface/token
Login successful


In [11]:
from huggingface_hub import list_models

# Authenticate
api = HfApi(token=os.getenv("HF_ACCESS_TOKEN"))

# Get all your model repos
your_username = "fmtmodels"  # <-- change this to your HF username
#models = list_models(author=your_username)
models = api.list_models(author=your_username)

#print(f"Found {len(list(models))} models in your account.")
# # Loop over each model and set to public
for model in models:
    repo_id = model.modelId
    print(f"Making {repo_id} public...")
    api.update_repo_visibility(repo_id=repo_id, private=False)

# print("✅ All done!")



Making fmtmodels/out-babylm_full_bpe_8k-6x6-mask_ee2000_em01-6849723 public...
Making fmtmodels/out-babylm_full_bpe_8k-6x6-mask_ee004_em10-6683311 public...
Making fmtmodels/out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8465077 public...
Making fmtmodels/out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8465082 public...
Making fmtmodels/out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8465084 public...
Making fmtmodels/out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8465085 public...
Making fmtmodels/out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8465086 public...
Making fmtmodels/out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8465087 public...
Making fmtmodels/out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8465089 public...
Making fmtmodels/out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8465090 public...
Making fmtmodels/out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8465091 public...
Making fmtmodels/out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8465093 public...
Making fmtmod

In [2]:
#Set Tokenizer Root as the folder that contains different tokenizer folders 
# Options - (relevant ones) - 
#   babylm_full_bpe_8k - Tokenizer for 10M models, vocab size 8k
#   babylm_full_bpe_100M_8k - Tokenizer for 100M models, vocab size 8k 
#Model's Relevant details can be found in the Model Table in the database (in rundata.xlsx)

TOKENIZER_ROOT = r"/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/data" 


#Set OUT_ROOT as the folder where all the models checkpoints are stored. Current setup expects the within OUT_ROOT to have a model with a folder_name, which contains a ckpt.pt file. ckpt.pt also contains the relevant model config details that get saved, so it is not explicitly required. To pick the right model, look into the database or rundata.xlsx file and pick the right model name (folder name) from OutputFolderName column(Database) or output_folder_name(from rundata.xlsx). 


MODELS_ROOT = r'/media/abishekthamma/Backup Plus/Projects/masters_thesis/ss-llm/nanoGPT/output_dump'

ROOT_ROOT = r'/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT'  

In [3]:
RESULTS_ROOT = os.path.join(ROOT_ROOT, "results")
SQL_DB = os.path.join(RESULTS_ROOT, "results.db")


def create_connection_cursor(db_file):
    """
    Create a database connection to the SQLite database specified by the db_file

    Args:
        db_file (str): database file

    Returns:
        Connection object or None
    """
    conn = sqlite3.connect(db_file)
    c = conn.cursor()
    return conn, c

conn, c = create_connection_cursor(SQL_DB)



In [13]:
# out_dir = "out-babylm_full_bpe_8k-6x6-mask_ee2000_em01-6849723"
# data_dir = "babylm_full_bpe_8k"
# out_dir = "out-babylm_full_bpe_8k-6x6-mask_ee004_em10-6683311"
# model, tokenizer = load_model_tokenizer(out_dir, data_dir)

def load_model(out_dir):
    ckpt_path = os.path.join(MODELS_ROOT, out_dir, 'ckpt.pt')
    print(f"Loading model from {ckpt_path}")
    # NANOGPT_ROOT = str(Path(__file__).parents[4])

    # Add if condition to check if inside server and if is, then add the path correctly. Default is local for now
    sys.path.append(ROOT_ROOT)
    #from model_HF import GPT, GPTConfig

    from FMT_GPT_Root.modeling_fleetingmemorytransformers import GPT
    from FMT_GPT_Root.configuration_fleetingmemorytransformers import GPTConfig

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    checkpoint = torch.load(ckpt_path, map_location=device)
    # Backward compatibility for new model args for QKV and FFW Adjustments
    if checkpoint["model_args"].get("wm_decay_length", None) is None:
        # wm_decay_length = block_size
        checkpoint["model_args"]["wm_decay_length"] = checkpoint["model_args"]["block_size"]
    # Setting head size as 3 times n_embd if not set already
    if checkpoint['model_args'].get('head_size_qkv', None) is None:
        checkpoint['model_args']['head_size_qkv'] = checkpoint['model_args']['n_embd']

    if checkpoint["model_args"].get("ffw_dim", None) is None:
        checkpoint["model_args"]["ffw_dim"] = 4 * checkpoint["model_args"]["n_embd"]

    gptconf = GPTConfig(**checkpoint["model_args"])

    model = GPT(gptconf)

    state_dict = checkpoint['model']
    unwanted_prefix = '_orig_mod.'
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    model.load_state_dict(state_dict)

    return model



def push_model_to_hub(model, model_name, model_dir):
    
    repo_id = f"fmtmodels/{model_name}"

    # 1) Ensure the repo exists
    try:
        create_repo(repo_id, private=True)
    except Exception:
        pass

    # 2) Upload everything in the folder
    api = HfApi()
    api.upload_folder(
        folder_path=model_dir,  # path to the folder containing the model files
        path_in_repo="",                # root of the repo
        repo_id=repo_id,
        repo_type="model",
        commit_message="Upload model + code"
    )


def delete_old_files():
    # Delete all files from FMT_GPT_Root where the file name is not in the list ["configuration_fleetingmemorytransformers.py", "modeling_fleetingmemorytransformers.py","__init__.py"]

    files_to_keep = ["configuration_fleetingmemorytransformers.py", "modeling_fleetingmemorytransformers.py","__init__.py"]
    for root, dirs, files in os.walk("FMT_GPT_Root"):
        for file in files:
            if file not in files_to_keep:
                os.remove(os.path.join(root, file))
                print(f"Deleted {file} from {root}")


# output_dir_name_list = ["out-babylm_full_bpe_8k-6x6-mask_ee2000_em01-6849723", "out-babylm_full_bpe_8k-6x6-mask_ee004_em10-6683311"]

# for output_dir_name in output_dir_name_list:
#     print(f"Processing {output_dir_name}")
#     model = load_model(output_dir_name)
#     model.save_pretrained("FMT_GPT_Root")
#     push_model_to_hub(model, output_dir_name, "FMT_GPT_Root")
#     delete_old_files()


#Step 1 - Load models important for the analysis

MODEL_LIST_QUERY = """SELECT DISTINCT ModelSurprisalScores.ModelID, Model.* FROM ModelSurprisalScores
JOIN Model on Model.ModelID = ModelSurprisalScores.ModelID
WHERE Model.NumLayers = 6 
AND Model.BatchSize = 32
AND ((Model.EchoicMemory=10 AND Model.MaskType="exponential_new" AND Model.MaskDecayRate=2) OR (Model.MaskType="Non" AND Model.CurriculumLearning=False)) 
AND Model.Dataset in ("babylm_full_bpe_8k", "babylm_full_bpe_100M_8k")  
AND Model.ModelID not in (5496427, 8456913)
ORDER BY Seed, BatchSize, Dataset, MaskType"""

model_list_df = pd.read_sql_query(MODEL_LIST_QUERY, conn)
model_list_df = model_list_df.sort_values(by=["OutputFolderName"])
important_model_list = sorted(model_list_df["OutputFolderName"].tolist())
print("Important Model List: ", len(important_model_list), important_model_list[:3])
model_list_df

#Push models to hub
for model_name in tqdm(important_model_list):
    print(f"Processing {model_name}")
    model = load_model(model_name)

     
    model.save_pretrained("FMT_GPT_Root", safe_serialization=False)
    # break
    push_model_to_hub(model, model_name, "FMT_GPT_Root")
    delete_old_files()


Important Model List:  40 ['out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8465077', 'out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8465082', 'out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8465084']


  0%|          | 0/40 [00:00<?, ?it/s]

Processing out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8465077
Loading model from /media/abishekthamma/Backup Plus/Projects/masters_thesis/ss-llm/nanoGPT/output_dump/out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8465077/ckpt.pt
Setting flash to False because wm_mask is enabled


/tmp/ipykernel_198021/212003239.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_path, map_location=device)


Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
number of parameters: 13.69M


...

Deleted config.json from FMT_GPT_Root
Deleted pytorch_model.bin from FMT_GPT_Root
Processing out-babylm_full_bpe_8k-6x6-nomask-6892221_s11111
Loading model from /media/abishekthamma/Backup Plus/Projects/masters_thesis/ss-llm/nanoGPT/output_dump/out-babylm_full_bpe_8k-6x6-nomask-6892221_s11111/ckpt.pt
number of parameters: 13.69M


pytorch_model.bin:   0%|          | 0.00/55.2M [00:00<?, ?B/s]

Deleted config.json from FMT_GPT_Root
Deleted pytorch_model.bin from FMT_GPT_Root
Processing out-babylm_full_bpe_8k-6x6-nomask-6892222_s1337
Loading model from /media/abishekthamma/Backup Plus/Projects/masters_thesis/ss-llm/nanoGPT/output_dump/out-babylm_full_bpe_8k-6x6-nomask-6892222_s1337/ckpt.pt
number of parameters: 13.69M


pytorch_model.bin:   0%|          | 0.00/55.2M [00:00<?, ?B/s]

Deleted config.json from FMT_GPT_Root
Deleted pytorch_model.bin from FMT_GPT_Root


In [37]:
remaining_models_df = pd.read_sql_query("SELECT * FROM Model", conn)
remaining_models_df = remaining_models_df.sort_values(by=["OutputFolderName"]).reset_index(drop=True)
remaining_models_df = remaining_models_df[~remaining_models_df["OutputFolderName"].isin(important_model_list)]

remaining_models_df

remaining_model_list = sorted(remaining_models_df["OutputFolderName"].tolist())
print("Remaining Model List: ", len(remaining_model_list), remaining_model_list[:3])

Remaining Model List:  219 ['out-babylm_full_bpe-4x4-nomask-5444724', 'out-babylm_full_bpe-6x6-mask_e002-5757736', 'out-babylm_full_bpe-6x6-mask_e100-5757737']


In [20]:
out_dir = "out-babylm_full_bpe_8k-6x6-mask_ee2000_em01-6849723"
data_dir = "babylm_full_bpe_100M_8k"
out_dir = "out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8465077"


def load_model(out_dir, device="cuda"):
    """
    Loads a pre-trained GPT model from a checkpoint file.

    Args:
        out_dir (str): The directory where the checkpoint file is located.
        device (torch.device): The device to load the model onto.

    Returns:
        GPT: The loaded GPT model.

    Raises:
        FileNotFoundError: If the checkpoint file is not found.
    """
    ckpt_path = os.path.join(MODELS_ROOT, out_dir, 'ckpt.pt')
    print(f"Loading model from {ckpt_path}")
    # NANOGPT_ROOT = str(Path(__file__).parents[4])

    # Add if condition to check if inside server and if is, then add the path correctly. Default is local for now
    sys.path.append(ROOT_ROOT)
    from model import GPT, GPTConfig

    checkpoint = torch.load(ckpt_path, map_location=device)

    # Backward compatibility for new model args for QKV and FFW Adjustments
    if checkpoint["model_args"].get("wm_decay_length", None) is None:
        # wm_decay_length = block_size
        checkpoint["model_args"]["wm_decay_length"] = checkpoint["model_args"]["block_size"]
    # Setting head size as 3 times n_embd if not set already
    if checkpoint['model_args'].get('head_size_qkv', None) is None:
        checkpoint['model_args']['head_size_qkv'] = checkpoint['model_args']['n_embd']

    if checkpoint["model_args"].get("ffw_dim", None) is None:
        checkpoint["model_args"]["ffw_dim"] = 4 * checkpoint["model_args"]["n_embd"]

    # print(checkpoint['model_args'])
    gptconf = GPTConfig(**checkpoint['model_args'])

    load_model = GPT(gptconf)

    state_dict = checkpoint['model']
    unwanted_prefix = '_orig_mod.'
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    load_model.load_state_dict(state_dict)
    load_model.eval()

    load_model = load_model.to(device)

    return load_model


def load_tokenizer(data_dir):
    """
    Load tokenizer for natural stories evaluation.

    Args:
        data_dir (str): The directory path where the tokenizer data is stored.

    Returns:
        tokenizer (Tokenizer): The loaded tokenizer object.

    Raises:
        NotImplementedError: If stoi/itos is not supported or found.

    """
    data_dir = os.path.join(TOKENIZER_ROOT, data_dir)
    meta_path = os.path.join(data_dir, "meta.pkl")
    load_meta = os.path.exists(meta_path)

    if load_meta:
        with open(meta_path, 'rb') as f:
            meta = pickle.load(f)
        if meta.get("custom_tokenizer", False):
            print(f"Loading custom tokenizer from {data_dir}")
            tokenizer = AutoTokenizer.from_pretrained(data_dir, use_fast=False)
        else:
            if meta.get("stoi", False):
                raise NotImplementedError("stoi/itos not supported yet")
            else:
                raise NotImplementedError("No stoi/itos found")
    else:
        print("No meta.pkl found")
        raise NotImplementedError("No meta.pkl found")

    if not tokenizer.eos_token:
        tokenizer.add_special_tokens({"eos_token": "</s>"})
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"  # Add if needed?
    return tokenizer


def load_model_tokenizer(out_dir, data_dir, device="cuda"):
    model = load_model(out_dir, device)
    tokenizer = load_tokenizer(data_dir)
    return model, tokenizer


model_2, tokenizer = load_model_tokenizer(out_dir, data_dir)

Loading model from /media/abishekthamma/Backup Plus/Projects/masters_thesis/ss-llm/nanoGPT/output_dump/out-babylm_full_bpe_100M_8k-6x6-mask_ee002_em10-8465077/ckpt.pt
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled


/tmp/ipykernel_198021/3396770675.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_path, map_location=device)


number of parameters: 13.69M
Loading custom tokenizer from /home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/data/babylm_full_bpe_100M_8k


In [21]:
print(model_2.lm_head.weight is model_2.transformer.wte.weight)

True


In [10]:
from model_HF import GPT, GPTConfig

#m1 = GPT.from_pretrained("fmtmodels/out-babylm_full_bpe_8k-6x6-mask_ee2000_em01-6849723")
m1 = GPT.from_pretrained(f"fmtmodels/{out_dir}", use_safetensors=False)


pytorch_model.bin:   0%|          | 0.00/56.8M [00:00<?, ?B/s]

Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
Setting flash to False because wm_mask is enabled
number of parameters: 16.77M


In [11]:
model_2.eval()
model_2.to("cuda")


sample_text = "The quick brown fox jumped over the lazy"

# Encode the input text
input_ids = tokenizer.encode(sample_text, return_tensors="pt")
output_ids = input_ids.clone()

#input ids as all but the last token
input_ids = input_ids[:, :-1].to("cuda")
output_ids = output_ids[:, 1:].to("cuda")


print([(i, repr(tokenizer.decode(i))) for i in input_ids[0]]) 

#Get Model Hidden States
with torch.no_grad():
    outputs = model_2(input_ids, targets = output_ids, hidden_states=True) #If you want Loss, pass expected tokens with target = tokens_to_predict


print("Logits: ", outputs[0].shape)
print("Loss", outputs[1])
print("Hidden States: ", [{f"Layer {i}": outputs[2][i].shape} for i in range(len(outputs[2]))])

[(tensor(258, device='cuda:0'), "'the'"), (tensor(1903, device='cuda:0'), "' quick'"), (tensor(2602, device='cuda:0'), "' brown'"), (tensor(3518, device='cuda:0'), "' fox'"), (tensor(7158, device='cuda:0'), "' jumped'"), (tensor(537, device='cuda:0'), "' over'"), (tensor(182, device='cuda:0'), "' the'"), (tensor(796, device='cuda:0'), "' la'")]
Logits:  torch.Size([1, 8, 8000])
Loss tensor(5.6927, device='cuda:0')
Hidden States:  [{'Layer 0': torch.Size([1, 8, 384])}, {'Layer 1': torch.Size([1, 8, 384])}, {'Layer 2': torch.Size([1, 8, 384])}, {'Layer 3': torch.Size([1, 8, 384])}, {'Layer 4': torch.Size([1, 8, 384])}, {'Layer 5': torch.Size([1, 8, 384])}, {'Layer 6': torch.Size([1, 8, 384])}]


In [12]:
# m1 = model 
m1.eval()
m1.to("cuda")

with torch.no_grad():
    outputs_m1 = m1(input_ids, targets=output_ids, hidden_states=True) #If you want Loss, pass expected tokens with target = tokens_to_predict
print("Logits: ", outputs_m1["logits"].shape)
print("Loss", outputs_m1["loss"])
print("Hidden States: ", [{f"Layer {i}": outputs_m1["hidden_states"][i].shape} for i in range(len(outputs_m1["hidden_states"]))])

Logits:  torch.Size([1, 8, 8000])
Loss tensor(5.6927, device='cuda:0')
Hidden States:  [{'Layer 0': torch.Size([1, 8, 384])}, {'Layer 1': torch.Size([1, 8, 384])}, {'Layer 2': torch.Size([1, 8, 384])}, {'Layer 3': torch.Size([1, 8, 384])}, {'Layer 4': torch.Size([1, 8, 384])}, {'Layer 5': torch.Size([1, 8, 384])}, {'Layer 6': torch.Size([1, 8, 384])}]


In [26]:
m1

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(8000, 384)
    (wpe): Embedding(256, 384)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=384, out_features=1152, bias=False)
          (c_proj): Linear(in_features=384, out_features=384, bias=False)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=384, out_features=1536, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1536, out_features=384, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=384, out_features=8000, bias=False)
)